# Health AI — train the food classifier

**What this does:** downloads the **Khana** dataset (~131K images, 80 Indian-dish classes,
500×500), builds a validated/deduped manifest, makes stratified splits, and fine-tunes a
mobile student model (`mobilenetv4_conv_medium`, timm) with AMP + EMA + per-epoch checkpoints
on Colab's free GPU.

**Licence honesty (read once):** Khana is licensed for **non-commercial research/education
only** and its images are web/delivery-platform sourced — the model trained here is a
**baseline/experiment**. Before any commercial release, re-train on self-collected,
consented photos (details in `ml/DATA_LICENSES.md` in the repo).

**Setup:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Checkpoints go to your Google Drive
every epoch, so a disconnect costs at most one epoch. Run the cells **top to bottom, in
order** — later cells depend on variables from earlier ones.

In [ ]:
# 1 — GPU sanity check
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True,
    ).stdout.strip())
else:
    print('NO GPU — set Runtime > Change runtime type > T4 GPU, then re-run')

In [ ]:
# 2 — get the repo (fresh or up-to-date) and install light dependencies
import os
from pathlib import Path
import subprocess

REPO = Path('/content/Health-AI')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/anshuldhiman-ai/Health-AI.git', str(REPO)],
        check=True,
    )
# Always align the clone with GitHub (handles our history rewrites too):
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(
    ['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True
)

ML_SRC = str(REPO / 'ml' / 'src')
os.environ['PYTHONPATH'] = ML_SRC

# torch + torchvision ship with Colab; we add timm (model zoo) + gdown (Drive dl)
subprocess.run(['pip', 'install', '-q', 'timm', 'gdown'], check=True)

import timm, torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
print('repo head:', subprocess.run(
    ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
    capture_output=True, text=True).stdout.strip())

## 3 — dataset
Khana ships as a Google Drive bundle of class folders (checked 2026-10-19:
[github.com/prabhuomkar/Khana](https://github.com/prabhuomkar/Khana), arXiv 2509.06006).
`gdown` pulls the whole folder (≈14 GB); extraction takes a few minutes. The files inside
have **no extensions** — that is expected, not corruption.

In [ ]:
# 3 — download Khana and extract
KHANA = Path('/content/khana')
if not KHANA.exists():
    subprocess.run(
        ['gdown', '--folder',
         'https://drive.google.com/drive/folders/1PWyJdkizw5ABBd8BIAnr_FZq91YZ2Uo0',
         '-O', str(KHANA), '--remaining-ok'],
        check=True,
    )

# extract archives the folder contains (runs only when found)
for archive in list(KHANA.rglob('*.zip')) + list(KHANA.rglob('*.tar.gz')) + list(KHANA.rglob('*.tgz')):
    if archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', str(archive), '-d', str(archive.with_suffix(''))], check=True)
    else:
        subprocess.run(['tar', '-xzf', str(archive), '-C', str(KHANA)], check=True)

def find_image_root(root: Path) -> Path:
    """Walk the tree to the dir whose subdirs are ~80 class folders
    (each holding files directly). Extension-agnostic."""
    best, best_count = root, 0
    for path in [root, *root.rglob('*')]:
        if not path.is_dir():
            continue
        subdirs = [d for d in path.iterdir() if d.is_dir()]
        if len(subdirs) < 20:
            continue
        populated = sum(1 for d in subdirs if any(f.is_file() for f in d.iterdir()))
        if populated >= 20 and populated > best_count:
            best, best_count = path, populated
    return best

DATA_ROOT = find_image_root(KHANA)
n_classes = len([d for d in DATA_ROOT.iterdir() if d.is_dir()])
print('image root:', DATA_ROOT)
print('classes found:', n_classes)
assert n_classes >= 80, 'unexpected class count — inspect /content/khana manually'

In [ ]:
# 4 — validate + dedupe + split (Sections 8.5 / 8.3 of CLAUDE.md)
# This step is the slow one: hashing + decoding 131K files takes ~10-20 minutes.

MANIFEST = '/content/khana/manifest_splits.csv'

def run(*cmd):
    result = subprocess.run(
        cmd, cwd=str(REPO / 'ml'),
        env={**os.environ, 'PYTHONPATH': os.environ['PYTHONPATH']},
    )
    if result.returncode != 0:
        raise RuntimeError(f'FAILED ({result.returncode}): {cmd}')

if not Path(MANIFEST).exists():   # idempotent: reuse existing manifest on re-run
    run('python', 'scripts/prepare_dataset.py',
        '--data-root', str(DATA_ROOT),
        '--out', '/content/khana/manifest.csv',
        '--source', 'khana')
    run('python', 'scripts/make_splits.py',
        '--manifest', '/content/khana/manifest.csv',
        '--out', MANIFEST, '--seed', '42')
print('manifest with splits ready:', MANIFEST)

In [ ]:
# 5 — mount Drive so checkpoints survive disconnects (Section 8.6)
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = '/content/drive/MyDrive/health-ai-runs/baseline'
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
print('checkpoints ->', RUN_DIR)

## 6 — train
Start with `SMOKE = True` (2 epochs) to prove the plumbing end-to-end. Then set it to
`False` and run again for the full ~30-epoch baseline. If Colab disconnects mid-run,
just re-run this cell — `--resume` picks up `last.pt` from Drive automatically.

**Day-1 rule (Section 8.7):** before pushing accuracy, also run the
`--no-pretrained` random-weights export check (see export notebook) so we know the
converter works.

In [ ]:
# 6 — train (checkpoint every epoch into Drive; resumed automatically)
SMOKE = True   # True: 2 epochs plumbing test | False: full baseline

cmd = [
    'python', 'scripts/train.py',
    '--splits', MANIFEST,
    '--out', RUN_DIR,
    '--model-name', 'mobilenetv4_conv_medium',
    '--image-size', '224',
    '--epochs', '2' if SMOKE else '30',
    '--warmup-epochs', '1' if SMOKE else '3',
    '--batch-size', '128',
    '--lr', '1e-3',
    '--num-workers', '2',
]
if (Path(RUN_DIR) / 'last.pt').exists():
    cmd += ['--resume', str(Path(RUN_DIR) / 'last.pt')]

result = subprocess.run(
    cmd, cwd=str(REPO / 'ml'),
    env={**os.environ, 'PYTHONPATH': os.environ['PYTHONPATH']},
)
if result.returncode != 0:
    raise RuntimeError('training failed above — scroll to the last traceback')

## 7 — evaluate the best EMA checkpoint on the test split
Report numbers exactly as computed (R7/R11): top-1 and top-5 separately, together with the
git commit id printed below. Never advertise calorie accuracy from classification accuracy
(Section 8.1).

In [ ]:
# 7 — test-split evaluation of the EMA weights in best.pt
import csv
import torch
from torch.utils.data import DataLoader
from ml_pipeline.models.classifier import build_model
from ml_pipeline.train.datasets import ManifestDataset
from ml_pipeline.train.train_loop import evaluate
from ml_pipeline.utils.seeding import seed_everything

seed_everything(42)
with open(MANIFEST, newline='') as f:
    classes = sorted({row['label'] for row in csv.DictReader(f)})

test_ds = ManifestDataset(Path(MANIFEST), 'test', classes, train=False)
loader = DataLoader(test_ds, batch_size=256, num_workers=2)

model = build_model('mobilenetv4_conv_medium', num_classes=len(classes), pretrained=False)
ckpt = torch.load(f'{RUN_DIR}/best.pt', map_location='cpu', weights_only=False)
ema_state = ckpt.get('ema') or ckpt['model']
model.load_state_dict(ema_state, strict=False)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metrics = evaluate(model.to(device), loader, device, len(classes))
print(f"classes: {len(classes)} | test images: {metrics['n']}")
print(f"top-1: {metrics['top1']:.4f} | top-5: {metrics['top5']:.4f}")
print('git commit of training code:', ckpt.get('meta', {}).get('git_commit'))

## After the baseline
1. **Error analysis (Section 8.7.9):** worst-recall classes and top confusion pairs —
   more photos of exactly those dishes beats more hyperparameter tuning.
2. **Export day-1 check:** convert a RANDOM-WEIGHTS student through `litert-torch` and run
   it inside the app's inference isolate BEFORE any accuracy push (Section 8.7).
3. **Model card (R11):** only real, script-produced numbers with the commit hash.
   Khana is non-commercial — the shipped model needs self-collected data (Section 8.3).
4. Never commit datasets or checkpoints (R8) — they live in Drive/DVC, not git.